# Test the UC4P2 compare API (with retrieval)

This notebook calls the same FastAPI the cluster will expose: **disambiguate → expand → decompose → retrieve (Cross-Dataset Discovery) → synthesize**.

It uses FastAPI `TestClient`, so you do **not** need a running uvicorn first. Fill `.env` with `DG_USERNAME` / `DG_PASSWORD` so search can get a Keycloak token.

Optional: `QUERY_DISAMBIGUATION_URL` (defaults to the Scayle dev ingress).

In [ ]:
import html
import json
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import HTML, JSON, Markdown, display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / ".env", override=False)
load_dotenv(ROOT.parent / "dg-query-disambiguation" / ".env", override=False)
os.environ.setdefault("AUTH_DISABLED", "true")
os.environ.setdefault("QDMR_BACKEND", "dspy")
os.environ.setdefault("QDMR_MODEL_NAME_OR_PATH", "")

from src.llm import configure_lm_from_env, llm_endpoint, selected_model


def show_json(obj, title="JSON"):
    display(Markdown(f"### {title}"))
    try:
        display(JSON(obj, expanded=False))
    except Exception:
        pass
    text = json.dumps(obj, indent=2, ensure_ascii=False, default=str)
    display(
        HTML(
            "<pre style='background:#0f172a;color:#e2e8f0;padding:16px 18px;"
            "border-radius:12px;overflow:auto;max-height:720px;"
            "font:13px/1.45 ui-monospace,SFMono-Regular,Menlo,monospace;"
            "box-shadow:inset 0 0 0 1px #1e293b'>"
            + html.escape(text)
            + "</pre>"
        )
    )


def show_pipeline(body):
    """Table of each pipeline step, then the full JSON."""
    steps = body.get("pipeline") or []
    rows = []
    for item in steps:
        color = {"ok": "#0f6e56", "partial": "#b45309", "halted": "#b91c1c", "failed": "#b91c1c"}.get(
            item.get("outcome"), "#334155"
        )
        rows.append(
            "<tr>"
            f"<td style='padding:8px 10px;white-space:nowrap'>{item.get('step')}</td>"
            f"<td style='padding:8px 10px'><b>{html.escape(str(item.get('title') or item.get('name')))}</b>"
            f"<br><span style='color:#64748b;font-size:12px'>{html.escape(str(item.get('name')))}</span></td>"
            f"<td style='padding:8px 10px;color:{color};font-weight:600'>{html.escape(str(item.get('outcome')))}</td>"
            f"<td style='padding:8px 10px'>{html.escape(str(item.get('summary') or ''))}</td>"
            "</tr>"
        )
    if rows:
        display(Markdown("### Pipeline trace"))
        display(
            HTML(
                "<table style='border-collapse:collapse;width:100%;font:14px/1.4 system-ui'>"
                "<tr style='background:#f1f5f9;text-align:left'>"
                "<th style='padding:8px 10px'>#</th><th style='padding:8px 10px'>Step</th>"
                "<th style='padding:8px 10px'>Outcome</th><th style='padding:8px 10px'>What happened</th></tr>"
                + "".join(rows)
                + "</table>"
            )
        )
    show_json(
        {
            "question": body.get("question"),
            "status": body.get("status"),
            "synthesized_answer": body.get("synthesized_answer"),
            "grounded_citations": body.get("grounded_citations"),
            "pipeline": steps,
        },
        "Traceable JSON",
    )

configured = configure_lm_from_env()
print("ROOT", ROOT)
print("LLM configured:", configured)
if configured:
    print("  model:", selected_model())
    print("  endpoint:", llm_endpoint())


## POST `/compare` in-process

`TestClient` mounts the app in this kernel. The JSON is the same payload you would get from curl or the cluster ingress.

In [ ]:
from fastapi.testclient import TestClient

from src.app.main import app

QUESTION = "How did a marriage look like in the 1800s compared to now?"

client = TestClient(app)
health = client.get("/health")
print("health", health.status_code, health.json())

response = client.post(
    "/compare",
    json={"query": QUESTION, "query_id": "nb-marriage", "k": 5},
)
print("compare", response.status_code)
body = response.json()
if response.status_code != 200:
    raise RuntimeError(body)
show_pipeline(body)


## Retrieval: search queries and passages

In [ ]:
print("search queries")
for slice_id, query in (body.get("search_queries") or {}).items():
    print(f"  [{slice_id}] {query}")

errors = body.get("retrieval_errors") or {}
if errors:
    print("\nretrieval errors")
    for slice_id, message in errors.items():
        print(f"  [{slice_id}] {message}")

print("\npassages")
for slice_id, items in (body.get("passages") or {}).items():
    print(f"\n=== {slice_id} ({len(items)} hits) ===")
    for item in items[:3]:
        snippet = (item.get("text") or item.get("text_snippet") or "")[:400]
        print(f"- {item.get('doc_id')}  sim={item.get('similarity')}  dataset={item.get('dataset_id')}")
        print(f"  {snippet}")

## Answer

In [ ]:
answer = body.get("synthesized_answer") or "(no answer)"
display(Markdown(answer))
print("\ncitations:", len(body.get("grounded_citations") or []))
for citation in (body.get("grounded_citations") or [])[:6]:
    print(f"  [{citation.get('citation_id')}] {citation.get('slice_id')} {citation.get('source_document')}")

## Inspect one graph node at a time

Stop after retrieval if you only want search hits.

In [ ]:
from src.orchestration.graph import stream_steps

for event in stream_steps(QUESTION, query_id="nb-steps", until="retrieve_slices"):
    name = event["step"]
    output = event["output"]
    print(f"\n===== {name} =====")
    if name == "disambiguate":
        print("status:", output.get("status"), "source:", output.get("disambiguation_source"))
        print("concept:", output.get("concept"), "type:", output.get("comparison_type"))
        print("slices:", output.get("slices"))
    elif name == "decompose":
        for step in output.get("qdmr") or []:
            print(step.get("instruction"))
    elif name == "retrieve_slices":
        print("queries:", output.get("search_queries"))
        print("errors:", output.get("retrieval_errors"))
        for slice_id, items in (output.get("passages") or {}).items():
            print(f"  {slice_id}: {len(items)} passages")
    else:
        keys = [key for key in output if key != "pipeline"]
        print("keys:", keys)

## Live cluster API

Calls `https://datagems-dev.scayle.es/language-pilot/compare` and shows the payload as formatted JSON.

In [ ]:
import urllib.error
import urllib.request

API = os.getenv(
    "LANGUAGE_PILOT_URL",
    "https://datagems-dev.scayle.es/language-pilot",
).rstrip("/")
payload = json.dumps({"query": QUESTION, "query_id": "nb-http", "k": 3}).encode()
request = urllib.request.Request(
    f"{API}/compare",
    data=payload,
    headers={"Content-Type": "application/json", "Accept": "application/json"},
    method="POST",
)
try:
    with urllib.request.urlopen(request, timeout=180) as http:
        live = json.loads(http.read().decode())
    show_pipeline(live)
except urllib.error.URLError as exc:
    print(f"No server at {API}: {exc.reason}")
